# Preview

Source orientation: printed pages 1-10, PDF pages 19-28. This notebook is an original visualization-first lesson inspired by the structure of *Modern Robotics: Mechanics, Planning, and Control* by Kevin M. Lynch and Frank C. Park. It does not quote or reproduce textbook prose, exercises, screenshots, or page crops.

## Chapter Question

What does a robot need to know about shape, motion, force, and choice before it can act?

This question is the thread for the chapter. The goal is not to memorize a list of formulas; it is to make the geometry inspectable. A robot mechanism is a physical machine, but the way we reason about it is through spaces, maps, constraints, tangents, and dual forces. The notebook keeps those objects visible. Every diagram and computation below is designed to answer a local question that a reader can check: what is moving, what is fixed, what map is being applied, what invariant should survive, and where can the model fail?

## Translation Guide

The source chapter or appendix is translated into computational language using these terms: configuration, task, motion, planning, control, contact, mobile base. The important conversions for this notebook are:

- Robot behavior becomes a chain from state representation to feasible motion.
- A mechanism is not just a drawing; it is a space of possible configurations.
- Planning and control are easier to reason about when their geometry is visible.

The central habit is to name the representation and the invariant at the same time. Coordinates are useful only when we know what geometric object they represent. A column of joint angles may describe a point on a torus, a homogeneous transform may describe a rigid frame, and a matrix rank may reveal an instantaneous loss of motion. The notebook therefore pairs each formula with a small experiment: a plot, a residual, an ellipsoid, a path, a graph, or a constraint surface.

## Route Through the Notebook

1. Establish the objects and conventions needed for preview.
2. Build a concept dependency map so definitions have visible structure.
3. Generate the main visual lab: robotics as geometry plus decisions.
4. Run a compact worked example that exposes an invariant.
5. Try an applied lab that changes a parameter and asks what should remain true.
6. Finish with sanity checks that assert artifact existence, image variation, and numerical margins.

This is a standalone course notebook. The PDF can be useful for historical context and exercises, but the lesson here is complete without it. Definitions are restated in fresh language, examples are computed from scratch, and visuals are generated by course-local code. When the notebook mentions a source span, it is a navigation reference rather than a dependency for comprehension.

## Visualization Storyboard

| Concept | Representation | Artifact | Inspection target |
| --- | --- | --- | --- |
| concept dependency map | directed graph | `artifacts/chapter-01-preview/figures/concept-dependency-map.png` | which definitions feed the chapter's computation |
| robotics as geometry plus decisions | course map and workspace sketch | `artifacts/chapter-01-preview/figures/robotics-preview-lab.png` | how mechanics, planning, and control become one pipeline |
| numeric invariant check | residual bar chart and JSON summary | `artifacts/chapter-01-preview/figures/robotics-preview-checks.png` | small residuals, positive margins, or rank changes |

The first visual is a dependency map. It is intentionally simple: before computing anything, the reader should see which concepts support which later moves. The second visual is the main lab for this chapter. It turns the chapter's core geometry into something that can be inspected rather than imagined. The third visual is a numeric check: a residual, rank, eigenvalue, path length, or comparable margin that can be tested after the figure is built.

## Working Principles

The most reliable way to learn robotics geometry is to move between three views. The first view is symbolic: equations name maps and constraints. The second view is numerical: a small instance exposes scale, rank, conditioning, and residuals. The third view is visual: geometry becomes a shape, path, ellipsoid, cone, graph, or surface. This notebook keeps all three views close together. If a symbolic statement is correct, the numerical experiment should produce the expected small residual or positive margin. If a visual is meaningful, it should make a specific invariant or failure mode easier to see.

For preview, the relevant failure modes are not side details; they are part of the concept. Singularities, chart boundaries, rank loss, time-scaling limits, contact mode changes, and wheel constraints are all examples of geometry asserting itself. A robust robotics model does not pretend those edges are absent. It names them, draws them, and then tests a small case so the reader can recognize the issue in later code.

## Applied Lab

Trace a simple manipulation task from configuration variables to a path and controller.

In the lab, vary one parameter at a time and predict the direction of change before running the code. For example, ask whether a rank should change, whether a path should lengthen, whether an eigenvalue should stay positive, whether a cone should widen, or whether a coordinate chart should approach a singular value. This prediction step is small, but it is what turns a figure into a mathematical instrument.

## Pitfalls To Watch

- Treating chapters as isolated formulas hides the shared geometry.
- Task space is not the same object as configuration space.

These pitfalls are deliberately operational. If a computation becomes confusing, check frame labels, units, rank, and the distinction between a physical object and its coordinates. Many robotics errors are not arithmetic mistakes; they are mismatches between a representation and the geometry it claims to encode.

## Takeaways

- Robotics is organized around spaces, maps, velocities, forces, and feedback.
- The later chapters reuse the same geometric vocabulary.

By the end of this notebook, the reader should be able to explain the chapter's main object, build a small computed example, interpret the generated visual artifact, and state at least one numerical sanity check that would catch an implementation mistake.

In [ ]:
from pathlib import Path
import json
import math
import sys

import numpy as np

HERE = Path.cwd().resolve()
BOOK_ROOT = next(
    parent for parent in [HERE, *HERE.parents]
    if (parent / "AGENTS.md").exists() and (parent / "Mordern Robotics.pdf").exists()
)
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, save_json
from utils.validation import assert_artifact, image_stats

print(f"BOOK_ROOT={BOOK_ROOT}")

In [ ]:
CHAPTER = {
  "number": 1,
  "slug": "chapter-01-preview",
  "title": "Preview",
  "notebook": "01-preview.ipynb",
  "printed_start": 1,
  "printed_end": 10,
  "pdf_start": 19,
  "pdf_end": 28,
  "part_slug": "part-01-geometric-foundations",
  "part_title": "Geometric Foundations",
  "theme": "configuration",
  "visual_focus": "robotics as geometry plus decisions",
  "visual_kind": "course map and workspace sketch",
  "artifact_stem": "robotics-preview",
  "inspection_target": "how mechanics, planning, and control become one pipeline",
  "question": "What does a robot need to know about shape, motion, force, and choice before it can act?",
  "terms": [
    "configuration",
    "task",
    "motion",
    "planning",
    "control",
    "contact",
    "mobile base"
  ],
  "translation": [
    "Robot behavior becomes a chain from state representation to feasible motion.",
    "A mechanism is not just a drawing; it is a space of possible configurations.",
    "Planning and control are easier to reason about when their geometry is visible."
  ],
  "lab": "Trace a simple manipulation task from configuration variables to a path and controller.",
  "pitfalls": [
    "Treating chapters as isolated formulas hides the shared geometry.",
    "Task space is not the same object as configuration space."
  ],
  "takeaways": [
    "Robotics is organized around spaces, maps, velocities, forces, and feedback.",
    "The later chapters reuse the same geometric vocabulary."
  ]
}

from utils.visuals import build_storyboard
storyboard = build_storyboard(CHAPTER)
storyboard

In [ ]:
from utils.visuals import build_chapter_visuals

outputs = build_chapter_visuals(CHAPTER)
for artifact in outputs['figures']:
    display_artifact(artifact, width=760)
outputs['metrics']

## Worked Example

The worked example below is intentionally compact and reusable. It checks a planar arm Jacobian, a rotation exponential/logarithm round trip, and the twist/wrench power pairing under a frame transform. These are small tests, but they exercise the same representation discipline used throughout the course.

Read the output as a preview of the course's recurring contract between geometry and computation. The endpoint is a point in the task plane produced by joint coordinates; the Jacobian rank tells which instantaneous task motions are locally available; manipulability compresses the arm's velocity directions into one scalar; the rotation round trip checks that the chosen coordinates still describe the same attitude; and the power pairing error checks that changing frames has not changed physical work rate. None of these numbers is interesting by itself. Each number is useful because it names what should remain invariant when a representation is changed.

In [ ]:
from utils.kinematics import planar_arm_points, planar_jacobian, manipulability_measure
from utils.lie import adjoint, se3_exp, se3_log, so3_exp, so3_log, transform_from, wrench_power

lengths = np.array([1.0, 0.75, 0.45])
theta = np.array([0.45, -0.8, 0.6])
points = planar_arm_points(lengths, theta)
J = planar_jacobian(lengths, theta)
R = so3_exp(np.array([0.2, -0.1, 0.35]))
round_trip = np.linalg.norm(so3_log(R) - np.array([0.2, -0.1, 0.35]))
T = transform_from(R, np.array([0.25, -0.1, 0.4]))
twist = np.array([0.1, -0.2, 0.3, 0.4, 0.2, -0.1])
wrench = np.array([0.3, 0.1, -0.2, 1.0, -0.4, 0.2])
power_before = wrench_power(twist, wrench)
power_after = wrench_power(adjoint(T) @ twist, np.linalg.inv(adjoint(T)).T @ wrench)
worked_example = {
    "endpoint": points[-1].round(4).tolist(),
    "jacobian_rank": int(np.linalg.matrix_rank(J)),
    "manipulability": float(manipulability_measure(J)),
    "rotation_log_round_trip": float(round_trip),
    "power_pairing_error": float(abs(power_before - power_after)),
}
worked_example

## Applied Lab

The applied lab changes behavior based on the chapter theme. It produces a small numerical summary that should agree with the visual lab: ranks should be plausible, residuals should be small, paths should have nonzero length, and positive-definite quantities should stay positive.

For this preview chapter, the lab uses the configuration theme and sweeps one joint of a two-link planar arm. The sweep is deliberately modest: its job is to make the idea of a configuration space feel concrete before the later chapters formalize it. When the arm stretches or folds, the task-space velocity directions collapse toward a line and manipulability drops. When the links sit away from those boundary poses, the endpoint can move in richer local directions. That single curve foreshadows a major habit of robotics: look for the shape of the space first, then interpret the controller, planner, or force calculation in that shape.

In [ ]:
from utils.control import pd_response
from utils.dynamics import two_link_mass_matrix
from utils.grasping import friction_cone, grasp_matrix
from utils.mobile import mecanum_wheel_matrix, unicycle_rollout
from utils.planning import cubic_time_scaling, dijkstra_grid, quintic_time_scaling

theme = CHAPTER["theme"]
if theme in {"configuration", "kinematics"}:
    sweep = np.linspace(-np.pi, np.pi, 90)
    values = [manipulability_measure(planar_jacobian(np.array([1.0, 0.8]), np.array([0.25, q]))) for q in sweep]
    lab_summary = {"theme": theme, "max_manipulability": float(np.max(values)), "min_manipulability": float(np.min(values))}
elif theme in {"rigid", "appendix"}:
    angles = np.linspace(0.0, 0.95 * np.pi, 60)
    residuals = [np.linalg.norm(so3_log(so3_exp(np.array([0.0, 0.0, a]))) - np.array([0.0, 0.0, a])) for a in angles]
    lab_summary = {"theme": theme, "max_rotation_residual": float(np.max(residuals))}
elif theme == "dynamics":
    eigs = np.array([np.linalg.eigvalsh(two_link_mass_matrix(q)) for q in np.linspace(-np.pi, np.pi, 80)])
    lab_summary = {"theme": theme, "minimum_mass_eigenvalue": float(eigs.min())}
elif theme == "planning":
    t = np.linspace(0, 1, 100)
    lab_summary = {"theme": theme, "cubic_midpoint": float(cubic_time_scaling(np.array([0.5]), 1.0)[0]), "quintic_midpoint": float(quintic_time_scaling(np.array([0.5]), 1.0)[0])}
elif theme == "contact":
    cone = friction_cone(np.array([0.0, 1.0]), 0.5)
    G = grasp_matrix(np.array([[1.0, 0.0], [-0.5, 0.86], [-0.5, -0.86]]), np.array([[-1.0, 0.0], [0.5, -0.86], [0.5, 0.86]]))
    lab_summary = {"theme": theme, "cone_samples": int(len(cone)), "grasp_rank": int(np.linalg.matrix_rank(G))}
elif theme == "mobile":
    controls = np.c_[np.ones(80) * 0.5, np.ones(80) * 0.3]
    path = unicycle_rollout(controls)
    lab_summary = {"theme": theme, "wheel_map_rank": int(np.linalg.matrix_rank(mecanum_wheel_matrix())), "final_pose": path[-1].round(4).tolist()}
else:
    lab_summary = {"theme": theme}

lab_summary

## Sanity Checks

The final cell asserts that the generated artifacts exist, are large enough to be useful, and have nontrivial pixel variation. It also stores a JSON summary under the chapter's artifact subtree so the notebook leaves a machine-checkable trace.

In [ ]:
artifact_stats = {}
for artifact in outputs['figures']:
    resolved = assert_artifact(artifact)
    stats = image_stats(resolved)
    assert stats['pixel_std'] > 2.0, f'low variation image: {resolved}'
    artifact_stats[resolved.name] = stats
assert_artifact(outputs['checks'], min_size=100)
sanity = {'chapter': CHAPTER['slug'], 'metrics': outputs['metrics'], 'worked_example': worked_example, 'lab_summary': lab_summary, 'image_stats': artifact_stats}
sanity_path = save_json(sanity, CHAPTER['slug'], 'checks', 'final-sanity.json')
assert_artifact(sanity_path, min_size=100)
display_artifact(sanity_path)
sanity